# Sentinel-2 Feature Extraction for Urban Heat Island Prediction — Rome

This notebook collects Sentinel-2 optical features over Rome and exports them as a
**GeoTIFF pinned to the same 30 m grid** as the Sentinel-1 and Landsat LST exports,
so the three-way merge on `x_utm` / `y_utm` is exact.

## What this fixes vs the original `estrazione_roma.py`

| # | Problem in old script | Fix here |
|---|---|---|
| 1 | Scene-level cloud filter only (`CLOUDY_PIXEL_PERCENTAGE < 5`) — whole images discarded for a small cloud, and contaminated pixels in "clean" scenes slip through | **Pixel-level SCL masking**: bad pixels masked individually, scene filter relaxed to 20% as a coarse pre-filter |
| 2 | `PIS`/`PGS` were **binary** (majority vote at 30 m) → near-zero model importance | **Fractional** land cover via `reduceResolution(mean)` → true 0–1 fraction of built-up / green within each 30 m cell |
| 3 | Raw bands `B2,B3,B4,B8` exported alongside the indices derived from them → redundant, split-count inflation | **Raw bands dropped** — only the physically interpretable indices kept |
| 4 | `NLI` (VIIRS night lights) included despite r=0.06 with LST and ~750 m native resolution | **NLI removed** |
| — | No `crsTransform` on some exports → grid offset risk | **Explicit `crsTransform`** identical to S1 / LST |

**Final feature set:** `NDVI, NDBI, NDWI, Albedo, PIS, PGS` (6 features).

## Setup
```bash
pip install earthengine-api rasterio numpy pandas
```

## 1. Authenticate and initialise

In [1]:
import ee
import numpy as np
import pandas as pd

# First run only: uncomment, authenticate, then comment back.
# ee.Authenticate()

PROJECT_ID = 'first-test-project-492808'   # <-- your GCP project id

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

print('Earth Engine initialised.')

Earth Engine initialised.


## 2. Configuration

The `CRS_TRANSFORM` is the **single source of truth** for the pixel grid. It was derived
from the Sentinel-1 GeoTIFF (`src.transform.c`, `src.transform.f` → 279750, 4653180) and
must be byte-for-byte identical across S1, S2 and LST exports. This is what guarantees the
three datasets line up pixel-for-pixel and the merge returns ~677k rows instead of 0.

In [2]:
# ── Area of interest — Rome bounding box (lon_min, lat_min, lon_max, lat_max) ──
AOI = ee.Geometry.Rectangle([12.35, 41.78, 12.65, 42.00])

# ── Time window — summer 2025, matches S1 composite and Landsat LST ───────────
START_DATE = '2025-06-01'
END_DATE   = '2025-09-01'

# ── Grid parameters — MUST match S1 and LST exactly ───────────────────────────
SCALE         = 30
CRS           = 'EPSG:32633'                       # UTM Zone 33N
CRS_TRANSFORM = [30, 0, 279750, 0, -30, 4653180]   # same anchor as S1 / LST

DRIVE_FOLDER  = 'EarthObservationProject'

print(f'AOI          : Rome bounding box')
print(f'Window       : {START_DATE} -> {END_DATE}')
print(f'Scale        : {SCALE} m')
print(f'CRS          : {CRS}')
print(f'crsTransform : {CRS_TRANSFORM}')

AOI          : Rome bounding box
Window       : 2025-06-01 -> 2025-09-01
Scale        : 30 m
CRS          : EPSG:32633
crsTransform : [30, 0, 279750, 0, -30, 4653180]


## 3. Pixel-level cloud masking (SCL)

Every `S2_SR_HARMONIZED` image carries a **Scene Classification Layer (SCL)** band — a
per-pixel label produced by the Sen2Cor processor. Instead of asking *"is the whole scene
clean?"* we ask *"is this individual pixel clean?"* and mask only the bad ones.

| SCL value | Class | Keep? |
|---|---|---|
| 1 | saturated / defective | mask |
| 3 | cloud shadow | mask |
| 8 | cloud — medium probability | mask |
| 9 | cloud — high probability | mask |
| 10 | thin cirrus | mask |
| 11 | snow / ice | mask |
| 2,4,5,6,7 | dark / vegetation / bare / water / unclassified | keep |

After masking, a pixel that was cloudy in one scene simply doesn't contribute to that
scene's value — the median composite then uses only the clear observations available for
each pixel independently.

In [3]:
def mask_s2_scl(image):
    """Mask clouds, shadows, cirrus and snow per-pixel using the SCL band."""
    scl = image.select('SCL')
    clear = (scl.neq(1)          # saturated / defective
             .And(scl.neq(3))    # cloud shadow
             .And(scl.neq(8))    # medium-probability cloud
             .And(scl.neq(9))    # high-probability cloud
             .And(scl.neq(10))   # thin cirrus
             .And(scl.neq(11)))  # snow / ice
    return image.updateMask(clear)

print('SCL masking function defined.')

SCL masking function defined.


## 4. Load the collection and build the median composite

The scene-level filter is relaxed to **20%**. Its only job now is to skip near-total
overcast scenes that would contribute almost nothing; the actual cloud removal is done
per-pixel by `mask_s2_scl`. Being stricter here (e.g. 5%) would needlessly throw away
scenes that are mostly clear over Rome.

In [11]:
s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(AOI)
      .filterDate(START_DATE, END_DATE)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))  # coarse pre-filter only
      .map(mask_s2_scl))

print('Scenes after filtering:', s2.size().getInfo())

# Median across all clear-sky observations, per pixel
s2_composite = s2.median().clip(AOI)
print('Composite bands:', s2_composite.bandNames().getInfo()[:6], '...')

Scenes after filtering: 51
Composite bands: ['B1', 'B2', 'B3', 'B4', 'B5', 'B6'] ...


## 5. Spectral indices and albedo

Only derived indices are kept — they encode the raw bands without the redundancy.

| Feature | Formula | Physical meaning |
|---|---|---|
| `NDVI` | (B8−B4)/(B8+B4) | Vegetation density → evapotranspiration / cooling |
| `NDBI` | (B11−B8)/(B11+B8) | Built-up intensity → heat absorption |
| `NDWI` | (B3−B8)/(B3+B8) | Surface water / moisture |
| `Albedo` | weighted band sum | Solar reflectivity → net heat gain |

`normalizedDifference` is a ratio, so it is unaffected by the harmonized collection's
reflectance scaling/offset. Albedo divides by 10000 to convert DN to reflectance, matching
the original script.

In [12]:
ndvi = s2_composite.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndbi = s2_composite.normalizedDifference(['B11', 'B8']).rename('NDBI')
ndwi = s2_composite.normalizedDifference(['B3', 'B8']).rename('NDWI')

albedo = s2_composite.expression(
    '(B2*0.1324 + B3*0.1287 + B4*0.1119 + B8*0.1171 + B11*0.0926 + B12*0.0634)',
    {
        'B2':  s2_composite.select('B2').divide(10000),
        'B3':  s2_composite.select('B3').divide(10000),
        'B4':  s2_composite.select('B4').divide(10000),
        'B8':  s2_composite.select('B8').divide(10000),
        'B11': s2_composite.select('B11').divide(10000),
        'B12': s2_composite.select('B12').divide(10000),
    }
).rename('Albedo')

print('Indices computed: NDVI, NDBI, NDWI, Albedo')

Indices computed: NDVI, NDBI, NDWI, Albedo


In [13]:
# NLI — VIIRS nighttime lights (monthly composite, ~500 m native resolution)
# Included to match the reference paper's feature set.
# Note: native resolution is coarse (~500 m); at 30 m many neighbouring pixels
# will share nearly identical values. SHAP attribution will reveal whether it
# contributes meaningfully after training.

viirs = (ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG')
         .filterBounds(AOI)
         .filterDate(START_DATE, END_DATE)
         .select('avg_rad')
         .median()
         .clip(AOI)
         .rename('NLI'))

print('NLI computed.')

NLI computed.


## 6. Fractional land cover — PIS and PGS

ESA WorldCover is a **10 m categorical** map (each pixel is exactly one class). The old
script took `worldcover.eq(50)` (a binary 0/1 image) and let the 30 m export pick the
**majority** class — collapsing everything back to binary and destroying the feature's
discriminative power.

The fix uses `reduceResolution(mean)`: for each 30 m output cell it averages the binary
10 m sub-pixels, giving the genuine **fraction** of that cell that is built-up (PIS) or
green (PGS) — a continuous 0–1 value.

**Why the explicit `reduceResolution` → `reproject` chain is needed:** `reduceResolution`
aggregates from the input's *native* projection (WorldCover's 10 m grid). The trailing
`reproject(crs, crsTransform)` then snaps the aggregated result onto our exact 30 m output
grid. Without this chain GEE may aggregate at the wrong scale and hand back binary values
again.

In [14]:
# WorldCover v200 — single global epoch, 10 m, EPSG:4326
worldcover = ee.ImageCollection('ESA/WorldCover/v200').first()

# Output projection = our exact 30 m grid
out_proj = ee.Projection(CRS).atScale(SCALE)

# PIS — fraction built-up (WorldCover class 50)
pis = (worldcover.eq(50)
       .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=1024)
       .reproject(crs=CRS, crsTransform=CRS_TRANSFORM)
       .rename('PIS'))

# PGS — fraction green (trees 10, shrubland 20, grassland 30, cropland 40)
pgs = (worldcover.remap([10, 20, 30, 40], [1, 1, 1, 1], 0)
       .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=1024)
       .reproject(crs=CRS, crsTransform=CRS_TRANSFORM)
       .rename('PGS'))

print('Fractional PIS and PGS computed (0-1 continuous).')

Fractional PIS and PGS computed (0-1 continuous).


## 7. Assemble and export — aligned to the S1 / LST grid

`NLI` is intentionally excluded (r=0.06 with LST, ~750 m native resolution → no usable
signal at 30 m). The export uses the shared `crsTransform`, so every S2 pixel centre falls
exactly on the corresponding S1 and LST pixel.

In [15]:
final_img = ee.Image.cat([ndvi, ndbi, ndwi, albedo, pis, pgs, viirs]).toFloat()

# Band order in the GeoTIFF == this list
FEATURE_ORDER = ['NDVI', 'NDBI', 'NDWI', 'Albedo', 'PIS', 'PGS', 'NLI']
final_img = final_img.select(FEATURE_ORDER)

print('Final feature bands:', final_img.bandNames().getInfo())

task = ee.batch.Export.image.toDrive(
    image=final_img,
    description='S2_Roma_Features_30m_v4',
    folder=DRIVE_FOLDER,
    fileNamePrefix='s2_roma_pixel_dataset_v4',
    scale=SCALE,
    crs=CRS,
    crsTransform=CRS_TRANSFORM,   # <-- pixel-perfect alignment with S1 / LST
    region=AOI,
    maxPixels=1e9,
    fileFormat='GeoTIFF'
)
task.start()
print('Export started — monitor at https://code.earthengine.google.com/tasks')
print('Output: Google Drive ->', DRIVE_FOLDER, '-> s2_roma_pixel_dataset_v4.tif')

Final feature bands: ['NDVI', 'NDBI', 'NDWI', 'Albedo', 'PIS', 'PGS', 'NLI']
Export started — monitor at https://code.earthengine.google.com/tasks
Output: Google Drive -> EarthObservationProject -> s2_roma_pixel_dataset_v4.tif


## 8. Read the GeoTIFF locally with rasterio

Run this **after** the export finishes and you've downloaded the `.tif` from Drive.
This reproduces the exact same pixel-to-row pattern as the S1 and LST notebooks
(`x_utm` / `y_utm` from the raster transform), so the merge keys line up.

In [16]:
import rasterio

S2_TIF_PATH = 's2_roma_pixel_dataset_v4.tif'   # adjust path if needed

with rasterio.open(S2_TIF_PATH) as src:
    stack  = src.read()                # shape: (n_bands, H, W)
    nodata = src.nodata
    print(f'CRS       : {src.crs}')
    print(f'Transform : {src.transform}')
    print(f'Shape     : {src.height} rows x {src.width} cols, {src.count} bands')
    print(f'NoData    : {nodata}')

    rows_idx, cols_idx = np.meshgrid(
        np.arange(src.height), np.arange(src.width), indexing='ij'
    )
    xs, ys = rasterio.transform.xy(
        src.transform, rows_idx.ravel(), cols_idx.ravel()
    )

# Build the dataframe — one column per band, plus UTM coordinates
data = {name: stack[i].ravel() for i, name in enumerate(FEATURE_ORDER)}
data['x_utm'] = np.array(xs)
data['y_utm'] = np.array(ys)
s2_df = pd.DataFrame(data)

# Drop nodata / masked pixels (NDVI is never legitimately NaN over land)
if nodata is not None:
    s2_df = s2_df[s2_df['NDVI'] != nodata]
s2_df = s2_df.dropna(subset=FEATURE_ORDER).reset_index(drop=True)

print(f'\nS2 pixels: {len(s2_df):,}')
s2_df.head()

CRS       : EPSG:32633
Transform : | 30.00, 0.00, 279750.00|
| 0.00,-30.00, 4653180.00|
| 0.00, 0.00, 1.00|
Shape     : 839 rows x 855 cols, 7 bands
NoData    : None

S2 pixels: 677,735


,NDVI,NDBI,NDWI,Albedo,PIS,PGS,NLI,x_utm,y_utm
0,0.304037,0.072054,-0.415314,0.106103,0.0,1.0,4.18,280515.0,4653165.0
1,0.334911,0.068923,-0.436943,0.094604,0.0,1.0,4.18,280545.0,4653165.0
2,0.370341,-0.010526,-0.481143,0.098444,0.0,1.0,4.18,280575.0,4653165.0
3,0.366675,-0.024875,-0.462777,0.101344,0.0,1.0,4.18,280605.0,4653165.0
4,0.393721,-0.046630,-0.476846,0.092789,0.0,1.0,4.18,280635.0,4653165.0


## 9. Sanity checks

Confirm the fixes actually took effect before trusting the data downstream.

In [17]:
# 1) PIS / PGS must now be CONTINUOUS, not binary
print('PIS unique values (first 10):', np.sort(s2_df['PIS'].unique())[:10])
print('PGS unique values (first 10):', np.sort(s2_df['PGS'].unique())[:10])
print('PIS  -> nunique:', s2_df['PIS'].nunique(), '| range:',
      round(s2_df['PIS'].min(), 3), '-', round(s2_df['PIS'].max(), 3))
print('PGS  -> nunique:', s2_df['PGS'].nunique(), '| range:',
      round(s2_df['PGS'].min(), 3), '-', round(s2_df['PGS'].max(), 3))
print('\n(If nunique is ~2 they are still binary — the reduceResolution chain failed.)')

# 2) Index ranges should be physically sensible
print('\nFeature summary:')
print(s2_df[FEATURE_ORDER].describe().round(3))

# 3) No raw bands, no NLI present
print('\nColumns:', s2_df.columns.tolist())

PIS unique values (first 10): [0.0000000e+00 7.9159818e-08 1.1556317e-07 1.5521010e-07 1.6517633e-07
 1.8436600e-07 1.9854527e-07 2.9791696e-07 3.4047000e-07 3.9124660e-07]
PGS unique values (first 10): [0.0000000e+00 7.1228961e-08 7.7940498e-08 7.9039488e-08 8.4767052e-08
 9.0483994e-08 1.0119088e-07 1.3197368e-07 1.4983931e-07 1.5291737e-07]
PIS  -> nunique: 228826 | range: 0.0 - 1.0
PGS  -> nunique: 232781 | range: 0.0 - 1.0

(If nunique is ~2 they are still binary — the reduceResolution chain failed.)

Feature summary:
             NDVI        NDBI        NDWI      Albedo         PIS         PGS  \
count  677735.000  677735.000  677735.000  677735.000  677735.000  677735.000   
mean        0.369      -0.002      -0.422       0.112       0.356       0.635   
std         0.200       0.127       0.162       0.025       0.425       0.428   
min        -0.512      -0.527      -0.827       0.014       0.000       0.000   
25%         0.221      -0.073      -0.528       0.096       0.000 

## 10. Save for the three-way merge

The S2 dataframe now shares the `x_utm` / `y_utm` keys with the S1 and LST dataframes,
so the merge is a plain inner join:

```python
merged = (s1_df
          .merge(s2_df,  on=['x_utm', 'y_utm'], how='inner')
          .merge(lst_df, on=['x_utm', 'y_utm'], how='inner'))
```

In [ ]:
s2_df.to_csv('rome_s2_features_v4.csv', index=False)
print('Saved: rome_s2_features_v4.csv', s2_df.shape)

Saved: rome_s2_features_v4.csv (677735, 9)


## 11. SRTM elevation

In [4]:
# SRTM Digital Elevation Model — 30m
# Native resolution is ~30m, same as our target grid — no resampling needed.

srtm = (ee.Image('USGS/SRTMGL1_003')
        .select('elevation')
        .clip(AOI)
        .toFloat()
        .rename('elevation'))

task = ee.batch.Export.image.toDrive(
    image=srtm,
    description='SRTM_Roma_Elevation_30m',
    folder='EarthObservationProject',
    fileNamePrefix='srtm_roma_30m',
    scale=SCALE,
    crs=CRS,
    crsTransform=CRS_TRANSFORM,   # same anchor as S1/S2/LST
    region=AOI,
    maxPixels=1e9,
    fileFormat='GeoTIFF'
)
task.start()
print('SRTM export started.')

SRTM export started.


In [5]:
import rasterio
import numpy as np
import pandas as pd

with rasterio.open('srtm_roma_30m.tif') as src:
    data = src.read(1)
    rows_idx, cols_idx = np.meshgrid(
        np.arange(src.height), np.arange(src.width), indexing='ij'
    )
    xs, ys = rasterio.transform.xy(
        src.transform, rows_idx.ravel(), cols_idx.ravel()
    )

srtm_df = pd.DataFrame({
    'elevation': data.ravel(),
    'x_utm': np.array(xs),
    'y_utm': np.array(ys)
}).dropna().reset_index(drop=True)

print(f'Elevation range: {srtm_df["elevation"].min():.0f} → {srtm_df["elevation"].max():.0f} m')
print(f'Pixels: {len(srtm_df):,}')

Elevation range: -5 → 288 m
Pixels: 677,735


In [6]:
srtm_df.to_csv('rome_srtm_elevation_30m.csv', index=False)
print('Saved: rome_srtm_elevation_30m.csv', srtm_df.shape)

Saved: rome_srtm_elevation_30m.csv (677735, 3)
